# Clase 005 — VS Code / Cursor para Python y Jupyter

**Parte 0 — Prerrequisitos** · VS Code Python docs.

> 🎯 Configurar VS Code como IDE serio para Python+Jupyter: intérprete por workspace, debugger gráfico, ruff, tests integrados.

> ⏱️ ~60 min

## ⚙️ Setup mínimo

**Extensiones obligatorias:**
- `ms-python.python` — soporte Python
- `ms-toolsai.jupyter` — notebooks nativos
- `charliermarsh.ruff` — linter + formatter
- `tamasfe.even-better-toml` — soporte `pyproject.toml`
- `eamodio.gitlens` — git superpoderes

Instala todas con:

```bash
code --install-extension ms-python.python
code --install-extension ms-toolsai.jupyter
code --install-extension charliermarsh.ruff
code --install-extension tamasfe.even-better-toml
code --install-extension eamodio.gitlens
```

## 1️⃣ Selector de intérprete (el bug invisible)

VS Code recuerda **un intérprete por workspace** en `.vscode/settings.json`. Si no lo configuras, usará el primero que encuentre — generalmente el del sistema. Resultado: `import` funciona en terminal pero no en VS Code (o al revés).

**Cómo configurarlo bien:**

```json
// .vscode/settings.json
{
  "python.defaultInterpreterPath": "${workspaceFolder}/.venv/bin/python",
  "python.terminal.activateEnvironment": true,
  "editor.formatOnSave": true,
  "[python]": {
    "editor.defaultFormatter": "charliermarsh.ruff",
    "editor.codeActionsOnSave": {
      "source.fixAll.ruff": "explicit",
      "source.organizeImports.ruff": "explicit"
    }
  }
}
```

En Windows: `"${workspaceFolder}/.venv/Scripts/python.exe"`.

In [ ]:
import sys
from pathlib import Path

print('Intérprete que ejecuta esta celda:')
print(f'  {sys.executable}')
print()

in_venv = sys.prefix != sys.base_prefix
print(f'¿Estás en un venv? {in_venv}')
if in_venv:
    print(f'  prefix: {Path(sys.prefix).name}')
else:
    print('⚠️  Selecciona un intérprete del venv del proyecto en VS Code.')

## 2️⃣ Debugger gráfico — el fin de los `print("AQUI 1")`

El debug por `print` es lento y no escala. VS Code te da:
- **Breakpoints** (F9): para la ejecución en la línea
- **Step over** (F10): siguiente línea
- **Step into** (F11): entra a la función
- **Step out** (Shift+F11): sale de la función
- **Variables** (panel izquierdo): valores actuales en el scope
- **Watch**: expresiones que evalúas en tiempo real
- **Call stack**: cómo llegaste aquí

Config mínima (`.vscode/launch.json`):

```json
{
  "version": "0.2.0",
  "configurations": [
    {
      "name": "Python: Archivo actual",
      "type": "debugpy",
      "request": "launch",
      "program": "${file}",
      "console": "integratedTerminal",
      "justMyCode": false
    }
  ]
}
```

`justMyCode: false` te deja entrar a código de librerías (útil cuando un error viene de pandas).

## 3️⃣ Notebooks nativos en VS Code

Mejor UX que Jupyter web para edición:
- Autocompletado con type hints reales (no solo nombres de variables)
- Hover muestra docstring de funciones de pandas/sklearn
- Debug de celda con breakpoint
- Git integrado (ves diffs por celda)
- Outline lateral con headers de markdown

Selector de kernel arriba a la derecha: elige el mismo intérprete del workspace para que `pip install` funcione consistente.

In [ ]:
# Demo: autocompletado funciona con type hints
from typing import Iterable

def promedio(xs: Iterable[float]) -> float:
    """Promedio aritmético — escribe `promedio(` y mira el hint."""
    xs = list(xs)
    return sum(xs) / len(xs) if xs else 0.0

print(promedio([1, 2, 3, 4, 5]))
print(promedio.__doc__)

## 4️⃣ ruff — un solo tool reemplaza 4

En 2026, **ruff** (Astral, Rust) sustituye al stack tradicional:
- ❌ `black` (formatter) → ✅ `ruff format`
- ❌ `isort` (import sort) → ✅ `ruff check --select I --fix`
- ❌ `flake8` (linter) → ✅ `ruff check`
- ❌ `pylint` (linter más estricto) → ✅ `ruff check --select PL`

Ventaja: 10–100× más rápido, 1 tool, 1 config.

Config recomendada (`pyproject.toml`):

```toml
[tool.ruff]
line-length = 100
target-version = "py312"

[tool.ruff.lint]
select = [
    "E",   # pycodestyle errors
    "F",   # pyflakes
    "I",   # isort
    "UP",  # pyupgrade (sintaxis moderna)
    "B",   # flake8-bugbear (bugs comunes)
    "N",   # pep8-naming
]
ignore = ["E501"]  # line-too-long lo deja al formatter

[tool.ruff.format]
quote-style = "double"
```

## 5️⃣ Tests integrados

Panel **Testing** (icono matraz). Con `pytest` instalado y tests en `tests/`:
- VS Code descubre automáticamente
- Click derecho → "Run Test" o "Debug Test"
- Output inline (verde/rojo) en el archivo
- Coverage opcional con `coverage.py` extension

Config (`pyproject.toml`):

```toml
[tool.pytest.ini_options]
testpaths = ["tests"]
python_files = "test_*.py"
addopts = "-v --tb=short"
```

## 6️⃣ ¿Cuándo Cursor en vez de VS Code?

**Cursor** = fork de VS Code con IA integrada (chat con contexto del proyecto, edición multi-archivo, autocompletado avanzado).

**Usa Cursor si:**
- Quieres pair programming con IA sin saltar a otra app
- Trabajas mucho en refactors o exploración de código nuevo
- Estás OK con pagar la suscripción

**Quédate con VS Code si:**
- Tu organización tiene políticas estrictas sobre IA
- Ya pagas Copilot y te alcanza
- No quieres dependencias adicionales

Ambos comparten extensiones — migrar es trivial.

## ✅ Checklist

- [ ] Mi VS Code apunta al intérprete del venv del proyecto
- [ ] Sé poner un breakpoint y debuggear sin `print`
- [ ] Edito notebooks en VS Code con autocompletado
- [ ] Tengo ruff configurado en `pyproject.toml`
- [ ] Sé correr tests desde el panel Testing

## 📝 Homework

Ver `README.md`. Repo con `.vscode/settings.json`, `pyproject.toml` con ruff, y screenshot del debugger en acción.

## 📖 Definiciones y características

**Workspace**

Concepto de VS Code = una carpeta (o conjunto de carpetas) con configuración asociada en `.vscode/settings.json`. La configuración del workspace **override** a la del usuario. Característica: pones `.vscode/` en git para que todos los colaboradores hereden la misma config.

**Intérprete Python**

Ejecutable concreto (`/path/to/.venv/bin/python`). VS Code recuerda **uno por workspace**. Es el origen del 90% de los "funciona en mi máquina" entre IDE y terminal.

**ruff**

Linter + formatter en un solo binario, escrito en Rust. Reemplaza black + isort + flake8 + (parte de) pylint con un único tool 10–100× más rápido. Config en `[tool.ruff]` de `pyproject.toml`.

**Breakpoint**

Marca en una línea (F9) que pausa la ejecución cuando llega ahí. Permite inspeccionar variables, paso a paso, evaluar expresiones — mil veces más eficiente que `print`.

**`launch.json`**

Config de debug de VS Code. Define perfiles: "debug archivo actual", "debug tests", "debug Django", etc. Cada perfil tiene su `program`, `args`, `env`, `justMyCode`.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| "Python interpreter is not selected" al abrir un .py | Workspace nuevo, VS Code no eligió uno. **Fix**: `Ctrl+Shift+P` → "Python: Select Interpreter" → elige el del `.venv` del proyecto. Guarda en `.vscode/settings.json` para que persista. |
| Format-on-save no aplica ruff aunque está instalado | Falta declarar ruff como formatter por default para Python. **Fix**: en `settings.json`, `"[python]": { "editor.defaultFormatter": "charliermarsh.ruff" }` y `"editor.formatOnSave": true`. |
| Debugger arranca pero se salta mis breakpoints | Estás corriendo el archivo (Ctrl+F5 = sin debug) en vez de debug (F5). O `justMyCode: true` está saltando código que vive en librerías que sí querías inspeccionar. |
| Tests no aparecen en el panel "Testing" | VS Code no detectó pytest. **Fix**: `Ctrl+Shift+P` → "Python: Configure Tests" → pytest → carpeta `tests`. O añade `[tool.pytest.ini_options] testpaths = ["tests"]` en `pyproject.toml`. |
| Cambié interpreter y los imports siguen rotos | VS Code cachea symbols del intérprete viejo. **Fix**: `Ctrl+Shift+P` → "Python: Restart Language Server". Si persiste, recarga la ventana (`Reload Window`). |

## ❓ Preguntas frecuentes

**❓ ¿VS Code o Cursor?**

Cursor = VS Code + IA integrada (chat con contexto del repo, edición multi-archivo). Si pagas Copilot o no te interesa IA, quédate en VS Code. Si quieres pair-programming con IA sin saltar a otra app, Cursor. Las extensiones son las mismas.

**❓ ¿Debo commitear `.vscode/`?**

**Sí** la parte compartida: `settings.json` (interpreter path relativo, formatter, etc.), `extensions.json` (recomendaciones). **No** lo personal: `.vscode/launch.json` con paths absolutos del tester.

**❓ ¿Notebook en VS Code o en JupyterLab?**

VS Code para escribir/refactorizar (autocomplete con type hints, debug por celda, git inline). JupyterLab cuando alguien necesita un navegador y no quiere instalar VS Code (alumno, demo en proyector).

**❓ ¿Para qué `justMyCode: false`?**

Por default, el debugger se salta código de librerías de terceros (numpy, pandas) — útil para no perderte. Pero a veces el bug viene **desde dentro de pandas** (datos malformados); con `false` puedes entrar a ver.

**❓ ¿Ruff reemplaza todo el stack? ¿No necesito black?**

Sí — `ruff format` es drop-in replacement de black (mismo output prácticamente). Mismo con isort (`ruff check --select I --fix`) y flake8 (`ruff check`). Único caso donde aún conviene black: si tu org ya tiene CI con black configurado y no quieres tocar.

## 🔗 Referencias

- [VS Code Python tutorial](https://code.visualstudio.com/docs/python/python-tutorial)
- [ruff docs](https://docs.astral.sh/ruff/)

➡️ **Siguiente:** [006 — Python: tipos, estructuras, control de flujo](../006-python-tipos-estructuras-control-de-flujo/README.md)

## ✅ Soluciones de los ejercicios

Intentá resolverlos vos primero; acá está una solución de referencia comentada.

> ⚙️ **Importante (clase de IDE):** no se puede *ejecutar* un IDE dentro de un notebook. Por eso las soluciones que siguen **no simulan** clicks de VS Code / Cursor: en su lugar validan la **configuración** (los JSON de `.vscode/`, el `pyproject.toml` de ruff) y reproducen la *lógica* que estarías depurando. Los pasos de UI (seleccionar intérprete, poner breakpoints, panel Testing) quedan descritos como guía; el código verifica el artefacto que producen.

### 🛠️ Walkthrough de configuración de VS Code / Cursor (esto es CONFIGURACIÓN, no ejecución)

VS Code y Cursor comparten el mismo formato de configuración por *workspace*: una carpeta `.vscode/` con archivos JSON que se commitean al repo para que todo el equipo herede la misma setup.

**`.vscode/settings.json`** — intérprete del venv, format-on-save con ruff, pytest y Jupyter:

```json
{
  "python.defaultInterpreterPath": "${workspaceFolder}/.venv/Scripts/python.exe",
  "python.terminal.activateEnvironment": true,
  "editor.formatOnSave": true,
  "[python]": {
    "editor.defaultFormatter": "charliermarsh.ruff",
    "editor.codeActionsOnSave": { "source.organizeImports": "explicit" }
  },
  "ruff.lint.enable": true,
  "python.testing.pytestEnabled": true,
  "python.testing.pytestArgs": ["tests"],
  "jupyter.notebookFileRoot": "${workspaceFolder}"
}
```

> En Windows el intérprete del venv vive en `.venv/Scripts/python.exe`; en Linux/macOS es `.venv/bin/python`.

**`.vscode/extensions.json`** — recomendaciones que VS Code ofrece instalar al abrir el repo:

```json
{
  "recommendations": [
    "ms-python.python",
    "ms-toolsai.jupyter",
    "charliermarsh.ruff",
    "eamodio.gitlens",
    "tamasfe.even-better-toml"
  ]
}
```

**Cómo aplicarlo:** creá la carpeta `.vscode/` en la raíz del repo, pegá esos dos archivos, y commiteá `settings.json` + `extensions.json` (no `launch.json` con paths absolutos). Al reabrir el proyecto, VS Code respeta el intérprete y aplica ruff al guardar. En **Cursor** es idéntico: mismo `.vscode/`, mismas extensiones.

**Ejercicio 1.** Seleccionar el intérprete del `.venv` del proyecto y verificarlo con `sys.executable`.

In [ ]:
# Solución Ej.1 — En VS Code/Cursor: Ctrl+Shift+P -> "Python: Select Interpreter" -> el del .venv.
# Aquí verificamos, headless, cuál es el intérprete que ejecuta ESTE kernel.
import sys, pathlib

exe = pathlib.Path(sys.executable)
print("Intérprete activo:", exe)
print("Versión:", sys.version.split()[0])

# El artefacto que "Select Interpreter" persiste es la ruta al ejecutable de Python:
assert exe.name.lower().startswith("python"), "sys.executable debe apuntar a un binario de Python"
assert sys.version_info[:2] >= (3, 8), "Se espera Python 3.8+"
print("OK: el kernel corre sobre un intérprete de Python válido.")

# Así se vería la línea que VS Code guarda en .vscode/settings.json para fijarlo por workspace:
settings_line = '"python.defaultInterpreterPath": "${workspaceFolder}/.venv/Scripts/python.exe"'
assert "python.defaultInterpreterPath" in settings_line
print("Ejemplo de settings.json:", settings_line)

**Ejercicio 2.** Debug paso a paso: tomar un script con un bug, poner breakpoint (F9) y navegar con F10/F11. Acá reproducimos el bug y mostramos qué revelaría el debugger.

In [ ]:
# Solución Ej.2 — En VS Code pondrías breakpoint (F9) dentro del for y correrías con F5,
# inspeccionando `total` e `i` paso a paso (F10). Acá reproducimos ese razonamiento en código.

def average_buggy(nums):
    total = 0
    # BUG: range(1, ...) se salta nums[0]; un breakpoint aquí mostraría i arrancando en 1
    for i in range(1, len(nums)):
        total += nums[i]
    return total / len(nums)

def average_fixed(nums):
    if not nums:
        raise ValueError("lista vacía")
    total = 0
    for i in range(len(nums)):   # FIX: empezar en 0
        total += nums[i]
    return total / len(nums)

datos = [10, 20, 30, 40]
print("buggy:", average_buggy(datos), "-> incorrecto (ignora el primer elemento)")
print("fixed:", average_fixed(datos))

assert average_buggy(datos) != 25, "el bug produce un promedio equivocado"
assert average_fixed(datos) == 25, "el fix debe promediar correctamente"
print("OK: el debugger habría mostrado i=1 en la primera vuelta, delatando el off-by-one.")

**Ejercicio 3.** Configurar ruff en `pyproject.toml` (`line-length = 100`, `select = ["E","F","I","UP"]`) y habilitar format-on-save. Validamos el TOML de ejemplo.

In [ ]:
# Solución Ej.3 — Este es el bloque que irías a pyproject.toml. Lo validamos parseándolo.
try:
    import tomllib          # stdlib en Python 3.11+
    loads = tomllib.loads
except ModuleNotFoundError:  # fallback por si el kernel es 3.10
    import tomli as tomllib  # type: ignore
    loads = tomllib.loads

pyproject = """
[tool.ruff]
line-length = 100

[tool.ruff.lint]
select = ["E", "F", "I", "UP"]
"""

cfg = loads(pyproject)
assert cfg["tool"]["ruff"]["line-length"] == 100
assert set(cfg["tool"]["ruff"]["lint"]["select"]) == {"E", "F", "I", "UP"}
print("OK: [tool.ruff] válido ->", cfg["tool"]["ruff"])

# Format-on-save se habilita en .vscode/settings.json (no en pyproject):
import json
vscode_settings = json.dumps({
    "editor.formatOnSave": True,
    "[python]": {"editor.defaultFormatter": "charliermarsh.ruff"},
})
s = json.loads(vscode_settings)
assert s["editor.formatOnSave"] is True
assert s["[python]"]["editor.defaultFormatter"] == "charliermarsh.ruff"
print("OK: format-on-save con ruff configurado.")

**Ejercicio 4.** Editar un notebook con autocompletado y type hints de pandas. Reproducimos el objeto sobre el que el autocompletado de VS Code opera.

In [ ]:
# Solución Ej.4 — El autocompletado de VS Code lee los type hints del objeto (aquí, un DataFrame).
# Headless verificamos que el objeto que "autocompletarías" existe y expone esos miembros.
import pandas as pd

df = pd.DataFrame({"ciudad": ["Lima", "Bogotá", "Quito"], "temp": [21.0, 14.5, 16.2]})

# Miembros que VS Code te sugeriría al escribir df.  (todos existen en el tipo DataFrame):
for attr in ["head", "describe", "groupby", "dtypes", "columns"]:
    assert hasattr(df, attr), f"df debería exponer .{attr} (lo que ofrece el autocompletado)"

print(df.dtypes)          # los dtypes son la info de tipos que alimenta los hints
resumen = df["temp"].mean()
assert round(resumen, 2) == 17.23
print("Temp media:", round(resumen, 2))
print("OK: el DataFrame expone los miembros que el autocompletado de VS Code sugiere.")

**Ejercicio 5.** Tests con un click: crear `tests/test_simple.py` con 2 tests (uno OK, uno FAIL) y correrlos desde el panel Testing. Acá los escribimos en un tempdir y verificamos el comportamiento sin romper el notebook.

In [ ]:
# Solución Ej.5 — Escribimos el archivo de tests en un directorio temporal, validamos su sintaxis,
# y ejecutamos las dos funciones a mano para MOSTRAR el resultado esperado (1 pasa, 1 falla),
# capturando el fallo para que el notebook siga corriendo. En VS Code esto sería el panel "Testing".
import ast, tempfile, pathlib

test_src = "\n".join([
    "def test_ok():",
    "    assert sum([1, 2, 3]) == 6",
    "",
    "def test_fail():",
    "    # Este está pensado para FALLAR (demuestra el rojo en el panel Testing)",
    "    assert sum([1, 2, 3]) == 7",
])

with tempfile.TemporaryDirectory() as d:
    tests_dir = pathlib.Path(d) / "tests"
    tests_dir.mkdir()
    test_file = tests_dir / "test_simple.py"
    test_file.write_text(test_src, encoding="utf-8")

    # 1) validar que el archivo es Python sintácticamente correcto (lo que pytest colectaría)
    tree = ast.parse(test_file.read_text(encoding="utf-8"))
    test_funcs = [n.name for n in tree.body if isinstance(n, ast.FunctionDef) and n.name.startswith("test_")]
    assert test_funcs == ["test_ok", "test_fail"], test_funcs
    print("Tests colectados:", test_funcs)

    # 2) ejecutar las funciones y clasificar resultados (sin depender de pytest instalado)
    ns = {}
    exec(compile(tree, str(test_file), "exec"), ns)
    resultados = {}
    for name in test_funcs:
        try:
            ns[name]()
            resultados[name] = "PASSED"
        except AssertionError:
            resultados[name] = "FAILED"

    print("Resultados:", resultados)
    assert resultados == {"test_ok": "PASSED", "test_fail": "FAILED"}
    print("OK: 1 test verde y 1 rojo, tal como los verías en el panel Testing de VS Code.")